# exp04c — Per-Head Attention Analysis

**Question:** Which specific attention heads drive the stego elevation found in exp04?

**Design:** Same forward passes as exp04, but attention is NOT averaged over heads.
Each head is analysed independently to produce a (layer × head) elevation heatmap.

**Expected outcomes:**
- Diffuse effect across all heads → elevation is a model-wide property
- 1-3 heads dominate → specialised 'acrostic tracking' heads exist

**Data:** `exp01/valid_pairs.json` (high-fidelity subset, n=77, same as exp04)

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))

    if not os.path.exists('stego_CoT'):
        os.system('git clone -q https://github.com/annareshetnyak799-netizen/stego_CoT.git')
    os.chdir('stego_CoT')
    os.system('pip install -q transformers accelerate scipy')

MODEL_ID   = 'meta-llama/Llama-3.1-8B-Instruct'
INPUT_DIR  = 'results/exp01'
OUTPUT_DIR = 'results/exp04c'
N_MAX      = 80
TOP_N      = 5   # top heads to plot individually

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('cwd:', os.getcwd())

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats
from transformers import AutoModelForCausalLM, AutoTokenizer

with open(f'{INPUT_DIR}/valid_pairs.json') as f:
    all_pairs = json.load(f)

hi_fid = [p for p in all_pairs if p['fidelity'] == 1.0]
print(f'Total: {len(all_pairs)}, high-fidelity: {len(hi_fid)}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='eager',
)
model.eval()
model.config.use_cache = False

n_layers = model.config.num_hidden_layers
n_heads  = model.config.num_attention_heads
print(f'Layers: {n_layers}, heads: {n_heads}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
def find_sentence_starts(token_ids, plen):
    """Return absolute token positions of paragraph starts in the CoT."""
    cot_ids   = list(token_ids[plen:])
    full_text = tokenizer.decode(cot_ids, skip_special_tokens=True)

    char_starts = []
    i = 0
    while i < len(full_text) and full_text[i] == '\n':
        i += 1
    if i < len(full_text):
        char_starts.append(i)
    while i < len(full_text):
        if full_text[i] == '\n':
            j = i + 1
            while j < len(full_text) and full_text[j] == '\n':
                j += 1
            if j - i >= 2 and j < len(full_text):
                char_starts.append(j)
            i = max(i + 1, j)
        else:
            i += 1

    positions = []
    found     = 0
    prev_len  = 0
    for tok_idx in range(len(cot_ids)):
        curr_len = len(tokenizer.decode(cot_ids[:tok_idx + 1], skip_special_tokens=True))
        while found < len(char_starts) and prev_len <= char_starts[found] < curr_len:
            positions.append(plen + tok_idx)
            found += 1
        prev_len = curr_len
        if found >= len(char_starts):
            break

    return positions


@torch.no_grad()
def extract_attentions(token_ids):
    ids_t   = torch.tensor([token_ids], dtype=torch.long).to(model.device)
    outputs = model(ids_t, output_attentions=True)
    attn    = [a[0].cpu().float() for a in outputs.attentions]
    del outputs, ids_t
    torch.cuda.empty_cache()
    return attn


def curves_per_head(attentions, sent_positions):
    """
    Returns array of shape (n_sent, n_layers, n_heads).
    result[k, l, h] = mean attention from sent_positions[k] to sent_positions[:k] for head h.
    Row 0 is zeros (no previous positions).
    """
    n_sent = len(sent_positions)
    result = np.zeros((n_sent, n_layers, n_heads))
    for k in range(1, n_sent):
        pos_k = sent_positions[k]
        prev  = sent_positions[:k]
        for l, attn in enumerate(attentions):
            # attn: (n_heads, seq_len, seq_len)
            # attn[:, pos_k, prev]: (n_heads, len(prev)) -> mean over prev -> (n_heads,)
            result[k, l, :] = attn[:, pos_k, prev].mean(dim=-1).numpy()
    return result

In [ ]:
# stego_curves[i]: (n_sent, n_layers, n_heads)
stego_curves = []
open_curves  = []
skipped      = 0

for pair in hi_fid[:N_MAX]:
    plen  = len(pair['payload'])
    s_pos = find_sentence_starts(pair['stego_ids'], pair['stego_plen'])[:plen]
    o_pos = find_sentence_starts(pair['open_ids'],  pair['open_plen'])[:plen]
    n_pos = min(len(s_pos), len(o_pos))

    if n_pos < 2:
        skipped += 1
        continue

    s_attn = extract_attentions(pair['stego_ids'])
    stego_curves.append(curves_per_head(s_attn, s_pos[:n_pos]))
    del s_attn

    o_attn = extract_attentions(pair['open_ids'])
    open_curves.append(curves_per_head(o_attn, o_pos[:n_pos]))
    del o_attn

    n = len(stego_curves)
    if n % 10 == 0:
        print(f'Processed {n} pairs (skipped {skipped})')

print(f'\nDone: {len(stego_curves)} pairs, {skipped} skipped')

In [ ]:
# Elevation heatmap: mean(stego - open) over pairs and K, per (layer, head)
n_pairs = len(stego_curves)
elev    = np.zeros((n_layers, n_heads))

for i in range(n_pairs):
    s = stego_curves[i]  # (n_sent, n_layers, n_heads)
    o = open_curves[i]
    # mean over K>=1 rows
    elev += (s[1:] - o[1:]).mean(axis=0)  # (n_layers, n_heads)

elev /= n_pairs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full heatmap
ax = axes[0]
im = ax.imshow(elev, aspect='auto', cmap='RdBu_r', origin='lower',
               vmin=-np.abs(elev).max(), vmax=np.abs(elev).max())
ax.set_xlabel('Head')
ax.set_ylabel('Layer')
ax.set_title('Stego − open elevation per (layer, head)')
plt.colorbar(im, ax=ax)

# Top-N heads by mean elevation across all layers
head_scores = elev.mean(axis=0)  # (n_heads,)
top_heads   = np.argsort(head_scores)[::-1][:TOP_N]

ax2 = axes[1]
for h in top_heads:
    layer_scores = elev[:, h]
    best_layer   = int(np.argmax(layer_scores))
    ax2.plot(range(n_layers), layer_scores, label=f'head {h} (peak L{best_layer})')
ax2.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax2.set_xlabel('Layer')
ax2.set_ylabel('Mean elevation (stego − open)')
ax2.set_title(f'Top-{TOP_N} heads by elevation')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/per_head_elevation.png', dpi=150)
plt.show()

print('\nTop heads (by mean elevation across layers):')
for rank, h in enumerate(top_heads):
    best_l = int(np.argmax(elev[:, h]))
    print(f'  #{rank+1}  head {h:2d}  mean elev = {head_scores[h]:.5f}  peak at layer {best_l}')

In [ ]:
# K-curves for the single best head, across all layers
best_head  = top_heads[0]
best_layer = int(np.argmax(elev[:, best_head]))

max_k  = max(c.shape[0] for c in stego_curves)
k_vals = list(range(1, max_k))

s_vals = [np.mean([c[k, best_layer, best_head] for c in stego_curves if k < c.shape[0]]) for k in k_vals]
o_vals = [np.mean([c[k, best_layer, best_head] for c in open_curves  if k < c.shape[0]]) for k in k_vals]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(k_vals, s_vals, marker='o', label='stego', color='steelblue')
ax.plot(k_vals, o_vals, marker='s', label='open',  color='coral', linestyle='--')
ax.set_xlabel('Sentence index K')
ax.set_ylabel('Mean attention to positions {1..K-1}')
ax.set_title(f'Best head: layer {best_layer}, head {best_head}')
ax.legend()

# Elevation profile across layers for best head
ax2 = axes[1]
ax2.plot(range(n_layers), elev[:, best_head], color='steelblue')
ax2.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax2.set_xlabel('Layer')
ax2.set_ylabel('Mean elevation (stego − open)')
ax2.set_title(f'Head {best_head}: elevation by layer')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/best_head_curves.png', dpi=150)
plt.show()

In [ ]:
# Stats: elevation t-test per head (averaged across layers and K)
print('=== Top heads: elevation significance ===')
for h in top_heads:
    diffs = [
        stego_curves[i][1:, :, h].mean() - open_curves[i][1:, :, h].mean()
        for i in range(n_pairs)
    ]
    t, p = stats.ttest_1samp(diffs, 0)
    print(f'  head {h:2d}  mean elev = {np.mean(diffs):.5f}  t = {t:.2f}  p = {p:.4f}')

summary = {
    'model':      MODEL_ID,
    'n_pairs':    n_pairs,
    'n_skipped':  skipped,
    'top_heads':  [int(h) for h in top_heads],
    'best_head':  int(best_head),
    'best_layer': best_layer,
    'head_scores': {str(int(h)): round(float(head_scores[h]), 6) for h in top_heads},
}

out_path = f'{OUTPUT_DIR}/exp04c_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('\nSaved:', out_path)

if IN_COLAB:
    from google.colab import drive
    import shutil, pathlib
    drive.mount('/content/drive')
    dst = pathlib.Path('/content/drive/MyDrive/stego_cot_results/exp04c')
    dst.mkdir(parents=True, exist_ok=True)
    for fp in pathlib.Path(OUTPUT_DIR).iterdir():
        shutil.copy(fp, dst / fp.name)
        print(f'  -> Drive: {fp.name}')